In [ ]:
from pathlib import Path

import numpy as np

from cardiac_electrophysiology import posterior_builder as builder
from cardiac_electrophysiology.ls_opt import logging as ls_opt_logging
from cardiac_electrophysiology.ls_opt import optimizer
from cardiac_electrophysiology.utils import analysis, visualization

In [ ]:
posterior_settings = builder.PosteriorBuilderSettings(
    paths=builder.Paths(
        mesh_path=Path("../data/mesh.vtu"),
        basis_vecs_path=Path("../data/basis_vecs.npy"),
        prior_mean_path=Path("../data/prior_mean_from_sde.npy"),
        ground_truth_path=Path("../data/ground_truth_from_sde.npy"),
        log_file_path=Path("../results/lsbip_logfile.log"),
    ),
    prior_parameters=builder.PriorParameters(
        kappa=0.005,
        tau=100,
        seed=0,
    ),
    eikonal_parameters=builder.EikonalParameters(
        solver_tolerance=1e-6,
        max_num_iterations=1000,
        max_value=1000,
        initial_site_ind=12650,
        longitudinal_velocity=3,
        transversal_velocity=1,
    ),
    observation_parameters=builder.ObservationParameters(
        num_observations=1000,
        noise_variance=1e-4,
        seed=0,
    ),
    logger_settings=builder.LoggerSettings(
        do_printing=False,
        write_mode="w",
    ),
)

optimizer_settings = optimizer.LBFGSConfig(
    maximum_num_iterations=10,
    relative_function_tolerance= 1e-6,
    relative_gradient_tolerance=1e-6,
    max_line_search_steps=100,
)
ls_opt_logger_settings = ls_opt_logging.LSOPTLoggerSettings(
    print_to_console=True,
    logfile_path= Path("../results/lsopt_logfile.log"),
)

In [ ]:
posterior_builder = builder.PosteriorBuilder(posterior_settings)
posterior, additional_output = posterior_builder.build(return_additional_data=True)
visualization.visualize_data_points(
    mesh=additional_output.pv_mesh,
    observation_inds=additional_output.observation_inds,
)

In [ ]:
initial_guess = np.zeros_like(additional_output.prior_mean_parameter)
ls_optimizer = optimizer.LBFGSOptimizer(optimizer_settings, ls_opt_logger_settings)
map_result = ls_optimizer.run(
    initial_guess=initial_guess,
    loss_function=posterior.evaluate_cost,
    gradient_function=posterior.evaluate_gradient,
)
print(f"MAP estimation success: {map_result.success}")
print(f"Status message: {map_result.status_message}")
np.save("../results/map_estimate.npy", map_result.result)
np.save("../results/map_loss_history.npy", map_result.loss_history)
np.save("../results/map_gradient_norm_history.npy", map_result.gradient_norm_history)

In [ ]:
map_parameter = np.load("../results/260422_map_estimate.npy")
analysis_data = analysis.compute_map_result_analysis(
    map_parameter=map_parameter,
    posterior=posterior,
    additional_output=additional_output,
)

In [ ]:
for data in (
    analysis_data.ground_truth_parameter,
    analysis_data.map_parameter,
    analysis_data.diff_lat_truth_prior,
    analysis_data.diff_lat_truth_map,
):
    visualization.visualize_scalar_field(
        mesh=additional_output.pv_mesh,
        scalar_field=data,
        circular=False,
    )